# Managing long conversations

**Module:** Module 3 — Production-Oriented Agent Design  
**Notebook:** `notebooks/module-3/3.2_managing_messages.ipynb`

## Purpose
Explore message summarization and message trimming/deletion to keep agent context within practical limits.

## Workflow
1. **Prepare the environment** — load required packages and local environment variables. Keep API keys in an ignored `.env` file; never paste credentials into notebook cells.
2. **Build the capability** — configure the model/agent, messages, tools, middleware or data source used in this lesson.
3. **Execute the example** — invoke or stream the agent and inspect the returned messages, tool calls, structured response or state.
4. **Validate behavior** — compare the result with the intended behavior, test an edge case, and check provider/service availability if execution fails.
5. **Reflect** — use the key learnings below to explain the design choices and limitations.

## Learning objectives
- Understand the concept demonstrated by this notebook and where it fits in an agent workflow.
- Explain the role of the main LangChain/LangGraph abstractions used in the implementation.
- Trace inputs through execution and interpret the resulting output.
- Identify operational concerns such as credentials, external services, state, safety and error handling.

## Key learnings
- Long conversations need deliberate context management, not simply unlimited message history.
- Human-in-the-loop checkpoints are a control boundary for actions with external side effects.
- Dynamic model/prompt/tool selection can reduce cost and tailor capabilities, but adds complexity.

## Prerequisites and execution notes
- Use the repository setup instructions in the root `README.md` and the supported Python version.
- Configure only the provider credentials needed by this notebook in your local `.env` file.
- Some cells call external APIs, local MCP services, audio devices or sample databases; those dependencies must be available.
- Model output is non-deterministic and provider model names/API behavior can change. Treat outputs as examples, not fixed expected strings.

---



### Implementation step 1

Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

from dotenv import load_dotenv

load_dotenv()

## Summarize messages

### Implementation step 3

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent; from langgraph.checkpoint.memory import InMemorySaver; from langchain.agents.middleware import SummarizationMiddleware. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent; from langgraph.checkpoint.memory import InMemorySaver; from langchain.agents.middleware import SummarizationMiddleware. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

### Implementation step 4

Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage, AIMessage; from pprint import pprint. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage, AIMessage; from pprint import pprint. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

### Implementation step 5

Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Inspects or prints the result so you can verify the output structure and follow the execution path.

print(response["messages"][0].content)

## Trim/delete messages

### Implementation step 7

Imports the libraries and APIs needed by the following example: from typing import Any; from langchain.agents import AgentState; from langchain.messages import RemoveMessage (and 3 more import statement(s)) Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from typing import Any; from langchain.agents import AgentState; from langchain.messages import RemoveMessage (and 3 more import statement(s)) Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

### Implementation step 8

Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

### Implementation step 9

Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

### Implementation step 10

Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Inspects or prints the result so you can verify the output structure and follow the execution path.

print(response["messages"][-1].content)